# SiPM Dark Count Rate Analysis

In this lab you will analyze real MASSIBO data from one silicon photomultiplier operated at liquid-nitrogen temperature.

You will estimate the dark count rate, inspect inter-arrival times, and identify simple candidates for correlated noise such as crosstalk-like and afterpulse-like events.

## Data Model

Each row in the `.npy` file is one trigger.

- column 0: timestamp in 62.5 MHz clock ticks
- columns 1 onward: ADC waveform samples

Use `1 tick = 16 ns` and `1 waveform sample = 16 ns`. The pulses in this dataset are negative-going.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'data' / 'massibo_example.npy').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from student import analysis

DATA_PATH = ROOT / 'data' / 'massibo_example.npy'
SAMPLE_SECONDS = 16e-9

## 1. Load The MASSIBO File

Complete `load_data` in `student/analysis.py`, then run this cell.

In [ ]:
timestamps, waveforms = analysis.load_data(DATA_PATH)

print(f'events: {len(timestamps)}')
print(f'waveforms shape: {waveforms.shape}')
print(f'first timestamp tick: {timestamps[0]}')
print(f'last timestamp tick: {timestamps[-1]}')

## 2. Inspect A Few Waveforms

This first plot checks the pulse polarity, baseline, and approximate pulse location.

In [ ]:
sample_axis_us = np.arange(waveforms.shape[1]) * SAMPLE_SECONDS * 1e6

fig, ax = plt.subplots(figsize=(7, 4))
for waveform in waveforms[:10]:
    ax.plot(sample_axis_us, waveform, alpha=0.7)
ax.set_xlabel('time within waveform [us]')
ax.set_ylabel('ADC counts')
ax.set_title('First 10 MASSIBO waveforms')
plt.show()

## 3. Convert Time And Estimate The Raw Rate

Complete the timing and rate functions in `student/analysis.py`. Do not assume the file lasted exactly 10 minutes; infer the live time from the timestamps.

The SiPM active area is `6 mm x 6 mm = 36 mm^2`, so you will also quote rates in `mHz/mm^2`.

In [ ]:
times_s = analysis.timestamps_to_seconds(timestamps)
live_time_s = analysis.compute_live_time(times_s)
interarrival_s = analysis.compute_interarrival_times(times_s)
rate_hz, rate_uncertainty_hz = analysis.compute_dark_count_rate(len(timestamps), live_time_s)

rate_density = analysis.convert_rate_to_mhz_per_mm2(rate_hz, area_mm2=36.0)
rate_density_uncertainty = analysis.convert_rate_to_mhz_per_mm2(rate_uncertainty_hz, area_mm2=36.0)

print(analysis.summarize_results(len(timestamps), live_time_s, rate_hz, rate_uncertainty_hz))
print(f'Dark count rate density: {rate_density:.2f} +/- {rate_density_uncertainty:.2f} mHz/mm^2')

## 4. Burst Tagging And Burst-Cleaned Rate

MASSIBO data may contain bursts: groups of many consecutive events separated by short time intervals. The DUNE-style tag method identifies burst candidates using two parameters:

- `dt_max_s = 0.1 s`, approximately 100 ms
- `n_min = 5` consecutive events

Complete `tag_burst_events` in `student/analysis.py`, then compute the burst-cleaned DCR by removing the tagged events from the event count.

In [ ]:
burst_mask = analysis.tag_burst_events(times_s, dt_max_s=0.1, n_min=5)
n_burst_events = np.sum(burst_mask)
n_nonburst_events = np.sum(~burst_mask)

clean_rate_hz, clean_rate_uncertainty_hz = analysis.compute_dark_count_rate(n_nonburst_events, live_time_s)
clean_rate_density = analysis.convert_rate_to_mhz_per_mm2(clean_rate_hz, area_mm2=36.0)
clean_rate_density_uncertainty = analysis.convert_rate_to_mhz_per_mm2(clean_rate_uncertainty_hz, area_mm2=36.0)

print(f'burst-tagged events: {n_burst_events}')
print(f'non-burst events: {n_nonburst_events}')
print(f'burst-cleaned DCR: {clean_rate_hz:.3f} +/- {clean_rate_uncertainty_hz:.3f} Hz')
print(f'burst-cleaned DCR density: {clean_rate_density:.2f} +/- {clean_rate_density_uncertainty:.2f} mHz/mm^2')

## 5. Inter-Arrival Time Distribution

For an ideal Poisson process, inter-arrival times follow an exponential distribution. Compare the full dataset with the burst-cleaned dataset.

The burst-cleaned inter-arrival times are computed after removing burst-tagged events, so long gaps across removed bursts may appear.

In [ ]:
clean_times_s = times_s[~burst_mask]
clean_interarrival_s = analysis.compute_interarrival_times(clean_times_s)

fig, ax = plt.subplots(figsize=(7, 4))
max_dt = np.percentile(interarrival_s, 95)
bins = np.linspace(0, max_dt, 50)

ax.hist(interarrival_s, bins=bins, density=True, alpha=0.45, label='full dataset')
ax.hist(clean_interarrival_s, bins=bins, density=True, alpha=0.45, label='burst-cleaned')

x = np.linspace(bins[0], bins[-1], 400)
ax.plot(x, rate_hz * np.exp(-rate_hz * x), color='black', lw=2, label='raw exponential')
ax.plot(x, clean_rate_hz * np.exp(-clean_rate_hz * x), color='tab:red', lw=2, label='cleaned exponential')

ax.set_xlabel('inter-arrival time [s]')
ax.set_ylabel('probability density')
ax.set_title('Inter-arrival times: full vs burst-cleaned')
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bins_us = np.logspace(0, 7, 70)

ax.hist(interarrival_s * 1e6, bins=bins_us, alpha=0.45, label='full dataset')
ax.hist(clean_interarrival_s * 1e6, bins=bins_us, alpha=0.45, label='burst-cleaned')

ax.set_xscale('log')
ax.set_xlabel('inter-arrival time [us]')
ax.set_ylabel('events')
ax.set_title('Short-delay structure: full vs burst-cleaned')
ax.legend()
plt.show()

Interpretation checkpoint: Where does the simple exponential model work well? Where does it visibly fail?

## 6. Burst-Cleaned Pulse Amplitudes

Complete the baseline and amplitude functions. Since the pulses are negative-going, the pulse amplitude is `baseline - minimum ADC`.

From this point on, use the burst-cleaned sample for amplitude and correlated-noise candidate counts.

In [ ]:
clean_waveforms = waveforms[~burst_mask]

baselines = analysis.estimate_baseline(clean_waveforms, n_pretrigger=40)
amplitudes = analysis.compute_negative_amplitudes(clean_waveforms, n_pretrigger=40)

print(f'burst-cleaned events: {len(clean_waveforms)}')
print(f'median baseline: {np.median(baselines):.1f} ADC')
print(f'median amplitude: {np.median(amplitudes):.1f} ADC')
print(f'95th percentile amplitude: {np.percentile(amplitudes, 95):.1f} ADC')

In [ ]:
full_amplitudes = analysis.compute_negative_amplitudes(waveforms, n_pretrigger=40)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(full_amplitudes, bins=50, alpha=0.35, label='full dataset')
ax.hist(amplitudes, bins=50, alpha=0.55, label='burst-cleaned')
ax.set_xlabel('negative pulse amplitude [ADC counts]')
ax.set_ylabel('events')
ax.set_title('Pulse amplitude distribution')
ax.legend()
plt.show()

## 7. Exploratory Amplitude Diagnostics

The next plots are useful diagnostics before applying simple crosstalk-like and afterpulse-like tags. They are not separate calibrations, but they help connect pulse size and event timing.

### Amplitude Versus Inter-Arrival Time

This plot uses the burst-cleaned sample. Each point corresponds to an event and its time since the previous burst-cleaned event.

In [ ]:
amplitudes_after_previous_event = amplitudes[1:]

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(
    clean_interarrival_s * 1e6,
    amplitudes_after_previous_event,
    s=18,
    alpha=0.7,
)

ax.set_xscale('log')
ax.set_ylim(0, 90)
ax.set_xlabel('time since previous event [us]')
ax.set_ylabel('negative pulse amplitude [ADC counts]')
ax.set_title('Burst-cleaned amplitude vs inter-arrival time')

ax.axvspan(1, 100, color='tab:orange', alpha=0.15, label='afterpulse-like window')
ax.axhline(
    2.5 * np.median(amplitudes),
    color='tab:red',
    linestyle='--',
    label='crosstalk-like threshold',
)

ax.legend()
plt.show()

### Amplitude Spectrum

This histogram is a simple amplitude-spectrum view of the burst-cleaned pulses. If gain peaks are visible, it resembles a finger plot, although dark-trigger data are usually less clean than dedicated LED or laser calibration data.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(
    amplitudes,
    bins=np.arange(0, max(220, np.max(amplitudes) + 5), 2),
    histtype='step',
    linewidth=1.8,
)

ax.set_xlim(0, 220)
ax.set_xlabel('negative pulse amplitude [ADC counts]')
ax.set_ylabel('events')
ax.set_title('Burst-cleaned amplitude spectrum')
plt.show()

### Integrated Charge Spectrum

A charge spectrum is often more useful than peak amplitude for SiPM gain studies. Here we compute a simple fixed-window, baseline-subtracted waveform integral around the pulse minimum region. This is only a quick diagnostic, not a full charge-reconstruction algorithm.

In [ ]:
pulse_window = slice(55, 85)
charge = np.sum(baselines[:, None] - clean_waveforms[:, pulse_window], axis=1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(charge, bins=80, histtype='step', linewidth=1.8)

ax.set_xlabel('fixed-window charge [ADC sample counts]')
ax.set_ylabel('events')
ax.set_title('Burst-cleaned charge spectrum')
plt.show()

## 8. Simple Correlated-Noise Candidate Tags

Use simple thresholds to tag candidate classes. These are not final detector-quality classifications; they are a compact way to connect waveform features and timing features to physical effects.

In [ ]:
crosstalk_threshold_adc = 2.5 * np.median(amplitudes)
crosstalk_mask = analysis.tag_crosstalk_candidates(amplitudes, crosstalk_threshold_adc)
afterpulse_mask = analysis.tag_afterpulse_candidates(clean_interarrival_s, min_delay_s=1e-6, max_delay_s=100e-6)

print(f'crosstalk-like threshold: {crosstalk_threshold_adc:.1f} ADC')
print(f'burst-cleaned crosstalk-like candidates: {np.sum(crosstalk_mask)}')
print(f'burst-cleaned afterpulse-like candidates, 1-100 us: {np.sum(afterpulse_mask)}')

Final checkpoint:

- Report the raw dark count rate with uncertainty in Hz and mHz/mm^2.
- Report the burst-cleaned dark count rate with uncertainty in Hz and mHz/mm^2.
- Comment on whether the inter-arrival times look purely Poissonian.
- Explain how burst, crosstalk-like, and afterpulse-like tags differ.
- Name one detector or electronics effect that could bias this analysis.